> To do: ricordati che potresti dover modificare la funzioni di feature e aggiungere drum_note_count  drum_note_density 

## Parte 2: Estraiamo le feature

### 2.1 Facciamo gli import e recuperiamo il dataset

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

import pretty_midi
from tqdm.auto import tqdm
from collections import Counter

eda_df_path=Path("/Users/mariavittoria/PycharmProjects/laboratorio_XAImusic/risultati/xmidi_midi_stats.csv")
eda_df=pd.read_csv(eda_df_path)

DATA_DIR = Path("../data/xmidi/XMIDI_Dataset")

midi_files = list(DATA_DIR.rglob("*.midi"))

print(f"Number of MIDI files: {len(midi_files)}")

Number of MIDI files: 108023


In [2]:
print(eda_df["n_drum_tracks"].value_counts().sort_index())

n_drum_tracks
0    14476
1    93547
Name: count, dtype: int64


### 2.2 Creiamo una funzione per estarre le feature

In [6]:
PITCH_CLASS_NAMES = [
    "Do", "Do_diesis", "Re", "Re_diesis",
    "Mi", "Fa", "Fa_diesis", "Sol",
    "Sol_diesis", "La", "La_diesis", "Si"
]


def extract_features(path):

    #carico i MIDI
    midi = pretty_midi.PrettyMIDI(str(path))

    #separo i strumenti melodici da quelli a percussione
    melodic_instruments = [
        inst for inst in midi.instruments
        if not inst.is_drum
    ]
    
    #prendo le note di tutti gli strumenti melodici
    melodic_notes = [
        note
        for inst in melodic_instruments
        for note in inst.notes
    ]

    #informazioni strutturali base:
    song_duration = midi.get_end_time()

    features = {
        "song_duration": song_duration,
        "n_instruments": len(midi.instruments),
        "has_drums": int(any(
            inst.is_drum for inst in midi.instruments)),
        "note_count": len(melodic_notes)
    }

    #controlli MIDI vuoti o senza note
    if song_duration <= 0 or len(melodic_notes) == 0:
        raise ValueError(
            "MIDI has zero duration or no melodic notes"
        )

    #creo vari array che contengono le informazioni sulle note:
    
    # l'altezza di ogni nota
    pitches = np.array(
        [note.pitch for note in melodic_notes],
        dtype=int
    )

    #la durata di ogni nota
    durations = np.array(
        [note.end - note.start for note in melodic_notes],
        dtype=float
    )
    
    #la velocità, cioè l'intensità con cui viene suonata ogni nota
    velocities = np.array(
        [note.velocity for note in melodic_notes],
        dtype=float
    )

    #il momento d'inizio di ogni nota
    onsets = np.array(
        [note.start for note in melodic_notes],
        dtype=float
    )

    #Feature riguardanti l'altezza delle note del brano:
    features["pitch_count"] = len(np.unique(pitches))
    features["pitch_mean"] = np.mean(pitches)
    features["pitch_std"] = np.std(pitches)
    features["pitch_min"] = np.min(pitches)
    features["pitch_max"] = np.max(pitches)
    features["pitch_range"] = np.ptp(pitches)

    # istogramma delle classi di altezza PCH (paper Yang & Lerch)
    pitch_classes = pitches % 12    #modulo il n di note, stessa nota ad ottave diverse sono la stessa nota

    
    pch = np.bincount(
        pitch_classes,
        minlength=12).astype(float)     #conto quante volte compare ogni altezza

    pch /= pch.sum()    #faccio la frequenza relativa

    for i, name in enumerate(PITCH_CLASS_NAMES):
        features[f"pch_{name}"] = pch[i]

    # feature ritmiche
    features["note_density"] = (
        len(melodic_notes) / song_duration
    )

    features["note_duration_mean"] = np.mean(durations)
    features["note_duration_std"] = np.std(durations)
    features["note_duration_median"] = np.median(durations)

    # IOI: intervalli tra onset distinti
    unique_onsets = np.unique(onsets)

    if len(unique_onsets) >= 2:
        ioi = np.diff(unique_onsets)
        features["ioi_mean"] = np.mean(ioi)
        features["ioi_std"] = np.std(ioi)
    else:
        features["ioi_mean"] = np.nan
        features["ioi_std"] = np.nan

    # feature dinamiche (intensità delle note)
    features["velocity_mean"] = np.mean(velocities)
    features["velocity_std"] = np.std(velocities)
    features["velocity_min"] = np.min(velocities)
    features["velocity_max"] = np.max(velocities)

    # feature relative alla polifonia
    onset_counts = Counter(onsets) #calcolo quante note iniziano ad ogni onset

    notes_per_onset = np.array(list(onset_counts.values()))

    features["mean_notes_per_onset"] = np.mean(
        notes_per_onset
    )

    features["simultaneous_onset_rate"] = np.mean(notes_per_onset > 1)  #in proporzione quanti onset hanno più note
    
    # strumenti
    programs = {
    inst.program
    for inst in midi.instruments
    if not inst.is_drum and len(inst.notes) > 0
    }
    
    # converte i numeri General MIDI in nomi leggibili
    instrument_names = sorted(
        pretty_midi.program_to_instrument_name(program)
        for program in programs
    )
    
    features["instruments_present"] = " | ".join(instrument_names)

    return features

### 2.3 Testiiamo prima un l'estrazione su 100 midi:

In [7]:

results = []
errors = []

for path in tqdm(midi_files[:100], desc="Extracting features"):
    try:
        features = extract_features(path)
        features["file"] = str(path)
        results.append(features)

    except Exception as e:
        errors.append({
            "file": str(path),
            "error": str(e)
        })

features_100 = pd.DataFrame(results)

print("MIDI elaborati:", len(features_100))
print("Errori:", len(errors))
print("Feature estratte:", len(features_100.columns) - 1)
print("Shape:", features_100.shape)
display(features_100.head())

Extracting features:   0%|          | 0/100 [00:00<?, ?it/s]

MIDI elaborati: 100
Errori: 0
Feature estratte: 35
Shape: (100, 36)


,song_duration,n_instruments,has_drums,note_count,pitch_count,pitch_mean,pitch_std,pitch_min,pitch_max,pitch_range,...,ioi_mean,ioi_std,velocity_mean,velocity_std,velocity_min,velocity_max,mean_notes_per_onset,simultaneous_onset_rate,instruments_present,file
0,272.647232,7,1,3076,44,54.294863,11.916315,28,81,53,...,0.253569,0.116883,93.732120,17.214429,40.0,126.0,2.864060,0.633147,Acoustic Bass | Acoustic Grand Piano | Acousti...,../data/xmidi/XMIDI_Dataset/XMIDI_warm_jazz_8U...
1,68.382396,2,0,312,23,62.256410,6.853197,45,74,29,...,0.402161,0.140897,80.000000,0.000000,80.0,80.0,1.846154,0.585799,Acoustic Grand Piano | Flute,../data/xmidi/XMIDI_Dataset/XMIDI_sad_classica...
2,153.466484,5,1,2876,24,50.239569,9.631527,30,68,38,...,0.181525,0.077871,80.539638,14.111449,50.0,118.0,3.407583,0.744076,Acoustic Bass | Acoustic Guitar (steel) | Stri...,../data/xmidi/XMIDI_Dataset/XMIDI_exciting_pop...
3,215.987532,5,1,2106,37,57.433998,14.592101,31,93,62,...,0.230164,0.160665,101.661918,17.438091,48.0,126.0,2.266954,0.571582,Acoustic Bass | Acoustic Grand Piano | Acousti...,../data/xmidi/XMIDI_Dataset/XMIDI_funny_countr...
4,163.424473,3,1,1229,7,80.336859,18.064268,17,89,72,...,0.141491,0.112061,71.254679,17.435159,50.0,98.0,1.070557,0.070557,Acoustic Bass | Acoustic Grand Piano,../data/xmidi/XMIDI_Dataset/XMIDI_happy_pop_SN...


In [8]:
# Summary statistics
display(features_100.describe().T)

# Basic consistency checks

pch_cols = [f"pch_{name}" for name in PITCH_CLASS_NAMES]  
assert np.allclose(features_100[pch_cols].sum(axis=1), 1.0)# PCH deve sommare a 1 per ogni canzone
assert (features_100["pitch_range"] >= 0).all()
assert (features_100["note_density"] > 0).all()
assert features_100["simultaneous_onset_rate"].between(0, 1).all()

print("Tutti i controlli superati!")

,count,mean,std,min,25%,50%,75%,max
song_duration,100.0,170.570374,63.335657,6.148511e+01,119.401336,166.617144,214.380388,301.512796
n_instruments,100.0,4.150000,1.445124,2.000000e+00,3.000000,4.000000,5.000000,8.000000
has_drums,100.0,0.900000,0.301511,0.000000e+00,1.000000,1.000000,1.000000,1.000000
note_count,100.0,2078.720000,1602.295078,1.270000e+02,865.250000,1919.000000,2515.250000,9336.000000
pitch_count,100.0,36.190000,14.652676,3.000000e+00,26.750000,36.000000,45.000000,74.000000
pitch_mean,100.0,57.543621,6.372813,3.853571e+01,54.840241,57.948921,61.423175,80.336859
pitch_std,100.0,11.641136,3.137858,2.509553e+00,9.909295,11.695410,13.307726,19.995511
pitch_min,100.0,30.150000,6.757525,1.700000e+01,26.000000,29.000000,32.000000,59.000000
pitch_max,100.0,82.890000,11.069017,4.400000e+01,78.000000,82.500000,89.000000,127.000000
pitch_range,100.0,52.740000,14.136863,7.000000e+00,45.000000,53.500000,61.000000,100.000000


Tutti i controlli superati!


In [9]:
feature_cols = [
    col for col in features_100.columns
    if col != "file"
]

n_unique = features_100[feature_cols].nunique().sort_values()

display(n_unique)

has_drums                    2
n_instruments                7
velocity_max                20
velocity_min                21
pitch_min                   28
pitch_max                   37
pitch_range                 41
note_duration_median        45
pitch_count                 50
instruments_present         52
pch_Re_diesis               68
pch_La_diesis               73
pch_Sol_diesis              78
pch_Fa                      78
pch_Do_diesis               81
pch_Fa_diesis               83
pch_Do                      84
pch_Si                      88
pch_Re                      90
velocity_std                91
pch_La                      91
pch_Mi                      91
pch_Sol                     93
velocity_mean               95
song_duration               97
simultaneous_onset_rate     98
note_count                  98
mean_notes_per_onset        98
note_duration_mean         100
note_duration_std          100
ioi_mean                   100
ioi_std                    100
pitch_st

### 2.4 Adesso passiamo all'estrazione su tutto il dataset diviso in batch:

In [11]:
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# Cartella in cui salvare le feature
output_dir = Path("../risultati/xmidi_features")
output_dir.mkdir(parents=True, exist_ok=True)

batch_size = 1000

all_errors = []

for start in tqdm(
    range(0, len(midi_files), batch_size),
    desc="Processing batches"
):

    end = min(start + batch_size, len(midi_files))
    batch_paths = midi_files[start:end]

    batch_number = start // batch_size

    output_file = output_dir / f"features_batch_{batch_number:04d}.csv"

    # Se il blocco è già stato salvato, lo saltiamo
    if output_file.exists():
        continue

    batch_results = []

    for path in batch_paths:
        try:
            features = extract_features(path)
            features["file"] = str(path)

            batch_results.append(features)

        except Exception as e:
            all_errors.append({
                "file": str(path),
                "error": str(e)
            })

    # Salvataggio del blocco
    if batch_results:
        pd.DataFrame(batch_results).to_csv(
            output_file,
            index=False
        )

print("Estrazione completata!")

Processing batches:   0%|          | 0/109 [00:00<?, ?it/s]

Estrazione completata!


In [4]:
output_dir = Path("../risultati/xmidi_features")
batch_files = sorted(output_dir.glob("features_batch_*.csv"))

features_df = pd.concat(
    [pd.read_csv(file) for file in batch_files],
    ignore_index=True
)

print("Numero MIDI:", len(features_df))
print("Numero feature:", len(features_df.columns) - 1)

display(features_df.head())

Numero MIDI: 108023
Numero feature: 35


,song_duration,n_instruments,has_drums,note_count,pitch_count,pitch_mean,pitch_std,pitch_min,pitch_max,pitch_range,...,ioi_mean,ioi_std,velocity_mean,velocity_std,velocity_min,velocity_max,mean_notes_per_onset,simultaneous_onset_rate,instruments_present,file
0,272.647232,7,1,3076,44,54.294863,11.916315,28,81,53,...,0.253569,0.116883,93.732120,17.214429,40.0,126.0,2.864060,0.633147,Acoustic Bass | Acoustic Grand Piano | Acousti...,../data/xmidi/XMIDI_Dataset/XMIDI_warm_jazz_8U...
1,68.382396,2,0,312,23,62.256410,6.853197,45,74,29,...,0.402161,0.140897,80.000000,0.000000,80.0,80.0,1.846154,0.585799,Acoustic Grand Piano | Flute,../data/xmidi/XMIDI_Dataset/XMIDI_sad_classica...
2,153.466484,5,1,2876,24,50.239569,9.631527,30,68,38,...,0.181525,0.077871,80.539638,14.111449,50.0,118.0,3.407583,0.744076,Acoustic Bass | Acoustic Guitar (steel) | Stri...,../data/xmidi/XMIDI_Dataset/XMIDI_exciting_pop...
3,215.987532,5,1,2106,37,57.433998,14.592101,31,93,62,...,0.230164,0.160665,101.661918,17.438091,48.0,126.0,2.266954,0.571582,Acoustic Bass | Acoustic Grand Piano | Acousti...,../data/xmidi/XMIDI_Dataset/XMIDI_funny_countr...
4,163.424473,3,1,1229,7,80.336859,18.064268,17,89,72,...,0.141491,0.112061,71.254679,17.435159,50.0,98.0,1.070557,0.070557,Acoustic Bass | Acoustic Grand Piano,../data/xmidi/XMIDI_Dataset/XMIDI_happy_pop_SN...


In [5]:
output_dir = Path("../risultati")
output_dir.mkdir(parents=True, exist_ok=True)

features_df.to_csv(
    output_dir / "xmidi_features_complete.csv",
    index=False
)

In [6]:
print("Numero righe:", len(features_df))

print("File duplicati:", features_df["file"].duplicated().sum())

print("Missing values totali:", features_df.isna().sum().sum())

print("Feature costanti:")
print(
    features_df.drop(columns="file")
    .nunique()
    .loc[lambda x: x <= 1]
)

Numero righe: 108023
File duplicati: 0
Missing values totali: 12
Feature costanti:
Series([], dtype: int64)


### 2.5 Verifichiamo cosa sono i valori mancanti

In [7]:
missing_per_feature = features_df.isna().sum()

print(missing_per_feature[missing_per_feature > 0])

ioi_mean    6
ioi_std     6
dtype: int64


In [8]:
rows_with_missing = features_df[
    features_df.isna().any(axis=1)
]

display(rows_with_missing[
    ["file"] + missing_per_feature[
        missing_per_feature > 0
    ].index.tolist()
])

,file,ioi_mean,ioi_std
8582,../data/xmidi/XMIDI_Dataset/XMIDI_exciting_roc...,NaN,NaN
13234,../data/xmidi/XMIDI_Dataset/XMIDI_funny_rock_D...,NaN,NaN
24984,../data/xmidi/XMIDI_Dataset/XMIDI_fear_country...,NaN,NaN
28012,../data/xmidi/XMIDI_Dataset/XMIDI_funny_rock_Y...,NaN,NaN
81699,../data/xmidi/XMIDI_Dataset/XMIDI_quiet_countr...,NaN,NaN
85067,../data/xmidi/XMIDI_Dataset/XMIDI_funny_pop_W3...,NaN,NaN


Ci accorgiamo che i valori mancanti riguardano solo una tipologia di feature, quella che riguarda la distanza tra gli onset di due note consecutive. Questo vuol dire che i brani coinvolti hanno solo note che iniziano nello stesso momento. 

In [9]:
missing_rows = features_df[
    features_df[["ioi_mean", "ioi_std"]].isna().any(axis=1)
]

display(
    missing_rows[
        [
            "file",
            "song_duration",
            "note_count",
            "mean_notes_per_onset",
            "ioi_mean",
            "ioi_std"
        ]
    ]
)

print("MIDI coinvolti:", len(missing_rows))

,file,song_duration,note_count,mean_notes_per_onset,ioi_mean,ioi_std
8582,../data/xmidi/XMIDI_Dataset/XMIDI_exciting_roc...,115.414989,1,1.0,NaN,NaN
13234,../data/xmidi/XMIDI_Dataset/XMIDI_funny_rock_D...,84.831986,7,7.0,NaN,NaN
24984,../data/xmidi/XMIDI_Dataset/XMIDI_fear_country...,74.873997,2,2.0,NaN,NaN
28012,../data/xmidi/XMIDI_Dataset/XMIDI_funny_rock_Y...,88.739552,3,3.0,NaN,NaN
81699,../data/xmidi/XMIDI_Dataset/XMIDI_quiet_countr...,90.456990,1,1.0,NaN,NaN
85067,../data/xmidi/XMIDI_Dataset/XMIDI_funny_pop_W3...,90.756360,3,3.0,NaN,NaN


MIDI coinvolti: 6


Andando a fare un ulteriore esplorazione ci rendiamo conto che i brani con questi valori mancanti hanno anche un numero molto ridotto di note all'interno, nonostante la durata suggerisca che sono comunque brani "reali". 

Si ipotizza allora che le note all'interno di questi brani possano essere solo note dovute alle percussioni che non vengono considerate nel nostro modello di notecount.

In [10]:
for path in missing_rows["file"]:
    midi = pretty_midi.PrettyMIDI(str(path))

    melodic_notes = [
        note
        for inst in midi.instruments
        if not inst.is_drum
        for note in inst.notes
    ]

    drum_notes = [
        note
        for inst in midi.instruments
        if inst.is_drum
        for note in inst.notes
    ]

    print("\nFile:", Path(path).name)
    print("Durata MIDI:", round(midi.get_end_time(), 2))
    print("Note melodiche:", len(melodic_notes))
    print("Note percussive:", len(drum_notes))

    print(
        "Onset melodici distinti:",
        len(set(note.start for note in melodic_notes))
    )

    print(
        "Note melodiche (pitch, start, end):",
        [
            (note.pitch, round(note.start, 3), round(note.end, 3))
            for note in melodic_notes[:10]
        ]
    )


File: XMIDI_exciting_rock_QX30KQSP.midi
Durata MIDI: 115.41
Note melodiche: 1
Note percussive: 944
Onset melodici distinti: 1
Note melodiche (pitch, start, end): [(60, np.float64(0.0), np.float64(0.032))]

File: XMIDI_funny_rock_DLJINT97.midi
Durata MIDI: 84.83
Note melodiche: 7
Note percussive: 875
Onset melodici distinti: 1
Note melodiche (pitch, start, end): [(48, np.float64(83.635), np.float64(84.265)), (60, np.float64(83.635), np.float64(84.832)), (64, np.float64(83.635), np.float64(84.832)), (72, np.float64(83.635), np.float64(84.832)), (76, np.float64(83.635), np.float64(84.832)), (79, np.float64(83.635), np.float64(84.832)), (84, np.float64(83.635), np.float64(84.832))]

File: XMIDI_fear_country_B2J6LGFO.midi
Durata MIDI: 74.87
Note melodiche: 2
Note percussive: 196
Onset melodici distinti: 1
Note melodiche (pitch, start, end): [(69, np.float64(0.252), np.float64(2.269)), (73, np.float64(0.252), np.float64(2.269))]

File: XMIDI_funny_rock_YTYF7OA0.midi
Durata MIDI: 88.74
Note 

Abbiamo verificato che era proprio come avevamo ipotizzato. 

I brani coinvolti sono composti prevalentemente da note percussive piuttosto che melodiche e le poche note melodiche presenti iniziano tutte allo stesso istante.

 Dato il ridotto numero di brani composti in questo modo nel nostro dataset decidiamo per ora di rimuoverli, ma consideriamo la possibilità di poter dover modificare la nostra funzione di estrazione di feature e dover aggiungere delle feature che analizzino meglio le tracce percussive

### 2.6 Rimuoviamo i 6 brani con missing values

In [11]:
features_cleaned = features_df.dropna(
    subset=["ioi_mean", "ioi_std"]
).copy()

print("MIDI originali:", len(features_df))
print("MIDI dopo la pulizia:", len(features_cleaned))
print("MIDI esclusi:", len(features_df) - len(features_cleaned))

# Verifichiamo che non rimangano missing values
assert features_cleaned.isna().sum().sum() == 0

MIDI originali: 108023
MIDI dopo la pulizia: 108017
MIDI esclusi: 6


In [12]:
output_dir = Path("../risultati")
output_dir.mkdir(parents=True, exist_ok=True)

features_cleaned.to_csv(
    output_dir / "xmidi_features_cleaned.csv",
    index=False
)

#### ora proviamo a utilizzare queste feature per classificare i brani
----
## Parte 3: Preparazione dei dati

In [13]:
dataset = features_cleaned.copy()

print(dataset.shape)
display(dataset.head())

(108017, 36)


,song_duration,n_instruments,has_drums,note_count,pitch_count,pitch_mean,pitch_std,pitch_min,pitch_max,pitch_range,...,ioi_mean,ioi_std,velocity_mean,velocity_std,velocity_min,velocity_max,mean_notes_per_onset,simultaneous_onset_rate,instruments_present,file
0,272.647232,7,1,3076,44,54.294863,11.916315,28,81,53,...,0.253569,0.116883,93.732120,17.214429,40.0,126.0,2.864060,0.633147,Acoustic Bass | Acoustic Grand Piano | Acousti...,../data/xmidi/XMIDI_Dataset/XMIDI_warm_jazz_8U...
1,68.382396,2,0,312,23,62.256410,6.853197,45,74,29,...,0.402161,0.140897,80.000000,0.000000,80.0,80.0,1.846154,0.585799,Acoustic Grand Piano | Flute,../data/xmidi/XMIDI_Dataset/XMIDI_sad_classica...
2,153.466484,5,1,2876,24,50.239569,9.631527,30,68,38,...,0.181525,0.077871,80.539638,14.111449,50.0,118.0,3.407583,0.744076,Acoustic Bass | Acoustic Guitar (steel) | Stri...,../data/xmidi/XMIDI_Dataset/XMIDI_exciting_pop...
3,215.987532,5,1,2106,37,57.433998,14.592101,31,93,62,...,0.230164,0.160665,101.661918,17.438091,48.0,126.0,2.266954,0.571582,Acoustic Bass | Acoustic Grand Piano | Acousti...,../data/xmidi/XMIDI_Dataset/XMIDI_funny_countr...
4,163.424473,3,1,1229,7,80.336859,18.064268,17,89,72,...,0.141491,0.112061,71.254679,17.435159,50.0,98.0,1.070557,0.070557,Acoustic Bass | Acoustic Grand Piano,../data/xmidi/XMIDI_Dataset/XMIDI_happy_pop_SN...


### 3.1 Recuperiamo le etichette

In [14]:
def extract_labels(file_path):
    parts = Path(file_path).stem.split("_")

    if len(parts) != 4 or parts[0] != "XMIDI":
        raise ValueError(f"Nome file non valido: {file_path}")

    return parts[1], parts[2]


dataset[["emotion", "genre"]] = pd.DataFrame(
    dataset["file"].apply(extract_labels).tolist(),
    index=dataset.index
)

In [15]:
print("Numero brani:", len(dataset))
print("Generi:", dataset["genre"].unique())
print("Emozioni:", dataset["emotion"].unique())

print("\nEtichette mancanti:")
print(dataset[["genre", "emotion"]].isna().sum())

display(dataset[["file", "genre", "emotion"]].head())

Numero brani: 108017
Generi: <StringArray>
['jazz', 'classical', 'pop', 'country', 'rock', 'traditional']
Length: 6, dtype: str
Emozioni: <StringArray>
[       'warm',         'sad',    'exciting',       'funny',       'happy',
       'angry',       'quiet',        'lazy', 'magnificent',        'fear',
    'romantic']
Length: 11, dtype: str

Etichette mancanti:
genre      0
emotion    0
dtype: int64


,file,genre,emotion
0,../data/xmidi/XMIDI_Dataset/XMIDI_warm_jazz_8U...,jazz,warm
1,../data/xmidi/XMIDI_Dataset/XMIDI_sad_classica...,classical,sad
2,../data/xmidi/XMIDI_Dataset/XMIDI_exciting_pop...,pop,exciting
3,../data/xmidi/XMIDI_Dataset/XMIDI_funny_countr...,country,funny
4,../data/xmidi/XMIDI_Dataset/XMIDI_happy_pop_SN...,pop,happy


In [16]:
print("Distribuzione dei generi:")
display(dataset["genre"].value_counts())

print("\nDistribuzione delle emozioni:")
display(dataset["emotion"].value_counts())

Distribuzione dei generi:


genre
rock           26705
pop            25581
country        23549
jazz           15862
classical      12660
traditional     3660
Name: count, dtype: int64


Distribuzione delle emozioni:


emotion
exciting       20947
warm           15090
happy          13291
romantic       12886
funny          12562
sad             9038
angry           8739
lazy            4622
quiet           4430
fear            3620
magnificent     2792
Name: count, dtype: int64

### 3.2 Separiamo le variabili

In [17]:
X = dataset.drop(columns=["file", "genre", "emotion"])

y_genre = dataset["genre"]
y_emotion = dataset["emotion"]

print("X:", X.shape)
print("y_genre:", y_genre.shape)
print("y_emotion:", y_emotion.shape)

X: (108017, 35)
y_genre: (108017,)
y_emotion: (108017,)


### 3.3 Controlliamo se sono presenti brani con lo stesso vettore di feature

In [18]:
print("File duplicati:", dataset["file"].duplicated().sum())

File duplicati: 0


In [19]:
duplicate_mask = X.duplicated(keep=False)

print(
    "Brani con vettore di feature condiviso:",
    duplicate_mask.sum()
)


Brani con vettore di feature condiviso: 13755


13755 brani hanno esattamente le stesse feature di almeno un altro brano all'interno del dataset. 

Questo rappresenta il 13% del nostro dataset, una percentuale significativa che ci porta a pensare che le feature che abbiamo definito non sono sufficientemente discriminative e dobbiamo aggiungerne altre.

### 3.4 Analizziamo i gruppi duplicati

In [20]:
duplicate_rows = dataset.loc[
    duplicate_mask,
    ["file", "genre", "emotion"]
].copy()
pd.set_option("display.max_colwidth", None)
display(duplicate_rows.head(20))

,file,genre,emotion
7,../data/xmidi/XMIDI_Dataset/XMIDI_warm_pop_FFY8JYNX.midi,pop,warm
14,../data/xmidi/XMIDI_Dataset/XMIDI_warm_country_LXGVTJT7.midi,country,warm
36,../data/xmidi/XMIDI_Dataset/XMIDI_fear_rock_MYPMG6ZQ.midi,rock,fear
57,../data/xmidi/XMIDI_Dataset/XMIDI_sad_country_Y7HDOGZ8.midi,country,sad
60,../data/xmidi/XMIDI_Dataset/XMIDI_romantic_rock_QM8BZKZJ.midi,rock,romantic
61,../data/xmidi/XMIDI_Dataset/XMIDI_funny_country_I21QKLLE.midi,country,funny
76,../data/xmidi/XMIDI_Dataset/XMIDI_exciting_rock_4WPO2XVW.midi,rock,exciting
81,../data/xmidi/XMIDI_Dataset/XMIDI_happy_classical_ZSBRF8TD.midi,classical,happy
87,../data/xmidi/XMIDI_Dataset/XMIDI_fear_rock_63RTDX0D.midi,rock,fear
99,../data/xmidi/XMIDI_Dataset/XMIDI_happy_country_4IBDFVV4.midi,country,happy


In [21]:
dataset["feature_group"] = X.groupby(
    list(X.columns),
    sort=False,
    dropna=False
).ngroup()

In [22]:
group_summary = dataset.groupby("feature_group").agg(
    n_files=("file", "size"),
    n_genres=("genre", "nunique"),
    n_emotions=("emotion", "nunique")
)

# Consideriamo solo i gruppi che contengono più di un MIDI
duplicate_groups = group_summary[
    group_summary["n_files"] > 1
]

print("Gruppi con vettori duplicati:", len(duplicate_groups))

print(
    "Gruppi con generi diversi:",
    (duplicate_groups["n_genres"] > 1).sum()
)

print(
    "Gruppi con emozioni diverse:",
    (duplicate_groups["n_emotions"] > 1).sum()
)

print("\nDimensioni dei gruppi:")
display(duplicate_groups["n_files"].describe())

Gruppi con vettori duplicati: 6281
Gruppi con generi diversi: 398
Gruppi con emozioni diverse: 1743

Dimensioni dei gruppi:


count    6281.000000
mean        2.189938
std         0.595686
min         2.000000
25%         2.000000
50%         2.000000
75%         2.000000
max        12.000000
Name: n_files, dtype: float64

In [23]:
# Prendiamo il gruppo con più file
largest_group = duplicate_groups["n_files"].idxmax()

pd.set_option("display.max_colwidth", None)

display(
    dataset.loc[
        dataset["feature_group"] == largest_group,
        ["file", "genre", "emotion"]
    ].head(20)
)

,file,genre,emotion
27914,../data/xmidi/XMIDI_Dataset/XMIDI_romantic_pop_3CX0VCNB.midi,pop,romantic
34088,../data/xmidi/XMIDI_Dataset/XMIDI_warm_pop_X94BABTH.midi,pop,warm
36977,../data/xmidi/XMIDI_Dataset/XMIDI_romantic_pop_L190H7RX.midi,pop,romantic
39992,../data/xmidi/XMIDI_Dataset/XMIDI_romantic_pop_VVQ8DFGA.midi,pop,romantic
52471,../data/xmidi/XMIDI_Dataset/XMIDI_happy_pop_P78P1U4S.midi,pop,happy
55544,../data/xmidi/XMIDI_Dataset/XMIDI_happy_pop_LBE8V76N.midi,pop,happy
62168,../data/xmidi/XMIDI_Dataset/XMIDI_warm_pop_1NGZBVVE.midi,pop,warm
69445,../data/xmidi/XMIDI_Dataset/XMIDI_warm_pop_6DZ90O2D.midi,pop,warm
70927,../data/xmidi/XMIDI_Dataset/XMIDI_romantic_pop_22QBNFBB.midi,pop,romantic
98154,../data/xmidi/XMIDI_Dataset/XMIDI_warm_pop_DSRQGSA1.midi,pop,warm


### 3.5 Analizziamo i gruppi di brani con stesse feature ma con etichette di genere e emozione discordanti

In [28]:
# Gruppi duplicati con emozioni discordanti
emotion_conflicts = duplicate_groups[
    duplicate_groups["n_emotions"] > 1
]

# Selezioniamo un gruppo
example_group = emotion_conflicts.index[0]

pd.set_option("display.max_colwidth", None)
# Visualizziamo i MIDI del gruppo
display(
    dataset.loc[
        dataset["feature_group"] == example_group,
        [
            "file",
            "genre",
            "emotion",
            "note_count",
            "song_duration",
            "pitch_mean"
        ]
    ]
)

,file,genre,emotion,note_count,song_duration,pitch_mean
57,../data/xmidi/XMIDI_Dataset/XMIDI_sad_country_Y7HDOGZ8.midi,country,sad,2364,258.151424,58.772843
28071,../data/xmidi/XMIDI_Dataset/XMIDI_warm_country_E6QLSUZM.midi,country,warm,2364,258.151424,58.772843
43568,../data/xmidi/XMIDI_Dataset/XMIDI_sad_country_MN40YZSU.midi,country,sad,2364,258.151424,58.772843
55437,../data/xmidi/XMIDI_Dataset/XMIDI_warm_country_90Y7TPCV.midi,country,warm,2364,258.151424,58.772843
79277,../data/xmidi/XMIDI_Dataset/XMIDI_sad_country_IQL8UCQD.midi,country,sad,2364,258.151424,58.772843


In [27]:
genre_conflicts = duplicate_groups[
    duplicate_groups["n_genres"] > 1
]

# Selezioniamo un gruppo
example_group = genre_conflicts.index[0]

pd.set_option("display.max_colwidth", None)
# Visualizziamo i MIDI del gruppo
display(
    dataset.loc[
        dataset["feature_group"] == example_group,
        [
            "file",
            "genre",
            "emotion",
            "note_count",
            "song_duration",
            "pitch_mean"
        ]
    ]
)

,file,genre,emotion,note_count,song_duration,pitch_mean
60,../data/xmidi/XMIDI_Dataset/XMIDI_romantic_rock_QM8BZKZJ.midi,rock,romantic,1709,122.03264,55.814511
71124,../data/xmidi/XMIDI_Dataset/XMIDI_romantic_rock_39OHC8G5.midi,rock,romantic,1709,122.03264,55.814511
87120,../data/xmidi/XMIDI_Dataset/XMIDI_romantic_country_KCOQJY8X.midi,country,romantic,1709,122.03264,55.814511


Ascoltando i brani che condividono tutte le feature ma in cui il genere o l'emozione sono discordanti ci rendiamo conto che i brani sono praticamente identici (_perlomeno ad un orecchio inesperto_). Notiamo solo che in alcuni brani viene eseguita la stessa melodia ma con uno strumento diverso